Copyright (c) Microsoft Corporation. All rights reserved.

Licensed under the MIT License.

# Redirecting User Output Streams

_**This notebook showcases how to capture the output of user scripts for non-managed runs and have it upload to the cloud.**_

---
---

## Table of Contents

1. [Introduction](#Introduction)
1. [Setup](#Setup)
    1. Validate Azure ML SDK installation
    1. Initialize workspace
    1. Set experiment
1. [Redirecting User Output Streams](#Logging)
    1. Redirecting the outputs of a run
    1. Redirecting the outputs of a child process inside a run

## Introduction

Capturing the output of the user script and streaming it to the cloud gives the users the ability to persist the logs of their run. Below we cover how to redirect the 

---

## Setup

Make sure you go through the [configuration notebook](../../../configuration.ipynb) first if you haven't.

### Validate Azure ML SDK installation and get version number for debugging purposes

In [ ]:
from azureml.core import Experiment, Workspace, Run
import azureml.core

# Check core SDK version number

print("This notebook was created using SDK version AZUREML-SDK-VERSION, you are currently running version", azureml.core.VERSION)

### Initialize workspace

Initialize a workspace object from persisted configuration.

In [ ]:
ws = Workspace.from_config()
print('Workspace name: ' + ws.name, 
      'Azure region: ' + ws.location, 
      'Subscription id: ' + ws.subscription_id, 
      'Resource group: ' + ws.resource_group, sep='\n')

### Set experiment
Create a new experiment (or get the one with the specified name).  An *experiment* is a container for an arbitrary set of *runs*. 

In [ ]:
experiment = Experiment(workspace=ws, name='redirect-user-outputs')

---

## Redirecting User Output Streams
In this section we will explore how users can toggle redirection of user output streams on and off as well as how to capture and stream the output of a child process. If redirected, the user output streams will be printed to `logs/user_log.txt`.

### Redirecting the outputs of a run

A *run* is a singular experimental trial.  This notebook focuses on runs created by calling `run = exp.start_logging()`. By default redirection will be turned on. To turn it off, create the run by setting `redirect_output_stream` to `False` as follows: `exp.start_logging(redirect_output_stream=False)`.

In [ ]:
import os

# start the run
with experiment.start_logging() as run:
    print("Hello world!")
    persist_run = run

# check user log
with open("logs/user_log.txt", "r") as user_log:
    print("Printing local file contents:")
    print(user_log.read())

# list filenames uploaded for this run
print(persist_run.get_file_names())

downloaded_file_path = "cloud_user_log.txt"
persist_run.download_file("logs/user_log.txt", downloaded_file_path)
with open(downloaded_file_path, "r") as stream:
    print("Printing contents of cloud file")
    print(stream.read())

# delete the log files
os.remove("logs/user_log.txt")
os.remove("cloud_user_log.txt")

### Redirecting the outputs of a child process in a run

In this subsection, we demonstrate how to capture the logs of a subprocess inside a file and upload it to the cloud.

In [ ]:
#subprocess is necessary to launch the child process
import subprocess
import os

with experiment.start_logging() as run:
    # need to open the file to write to it
    # one can pass in another file inside logs/ and it will also be tracked
    with open("logs/user_log.txt", "w") as file:
        subprocess.check_call(['python', 'child_process.py'], stdout=file, stderr=file)
    child_process_run = run

with open("logs/user_log.txt", "r") as user_log:
    print("Printing local file contents:")
    print(user_log.read())

# list filenames uploaded for this run
print(child_process_run.get_file_names())

downloaded_file_path = "cloud_user_log.txt"
child_process_run.download_file("logs/user_log.txt", downloaded_file_path)
with open(downloaded_file_path, "r") as stream:
    print("Printing contents of cloud file:")
    print(stream.read())

os.remove("logs/user_log.txt")
os.remove("cloud_user_log.txt")